# Churn Prediction Inteligente - Interpretabilidad con SHAP

En este notebook se interpreta el modelo principal del proyecto usando SHAP. Se selecciona la Regresion Logistica porque obtuvo el mejor F1-score y el mejor ROC-AUC. Aunque XGBoost obtuvo mejor Accuracy, en un problema de churn son especialmente importantes F1 y ROC-AUC, ya que ayudan a identificar mejor a los clientes con riesgo de abandono.

## 1. Importacion de librerias

In [ ]:
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split

# Configuracion general para graficas de SHAP.
shap.initjs()
plt.rcParams["figure.figsize"] = (10, 6)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "processed" / "telco_churn_limpio.csv").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("No se encontro processed/telco_churn_limpio.csv")
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTADOS_DIR = PROJECT_ROOT / "resultados"
RESULTADOS_DIR.mkdir(exist_ok=True)

## 2. Carga del dataset limpio

In [ ]:
ruta_dataset = PROJECT_ROOT / "processed" / "telco_churn_limpio.csv"
df = pd.read_csv(ruta_dataset)

display(df.head())
print("Filas y columnas:", df.shape)
print("\nInformacion del dataset:")
df.info()
print("\nDistribucion de Churn:")
print(df["Churn"].value_counts())

## 3. Carga del modelo seleccionado

In [ ]:
ruta_modelo = PROJECT_ROOT / "modelos" / "modelo_regresion_logistica.pkl"
pipeline = joblib.load(ruta_modelo)

# El archivo guardado contiene el Pipeline completo: preprocesamiento + modelo.
preprocesador = pipeline.named_steps["preprocesamiento"]
modelo = pipeline.named_steps["modelo"]

print("Pipeline cargado desde:", ruta_modelo)
print("Pasos del Pipeline:", list(pipeline.named_steps.keys()))
print("Modelo seleccionado:", modelo)

## 4. Preparacion de variables dependientes e independientes

In [ ]:
# Churn es la variable objetivo.
# customerID no se usa para entrenar porque solo identifica al cliente.
# Las demas columnas se usan como variables predictoras.
y = df["Churn"]
X = df.drop(["customerID", "Churn"], axis=1)

print("Variable objetivo:", y.name)
print("Cantidad de variables predictoras:", X.shape[1])
print("Variables predictoras:")
print(X.columns.tolist())

## 5. Division de datos en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Distribucion de Churn en prueba:")
print(y_test.value_counts(normalize=True))

## 6. Transformacion de datos usando el preprocesamiento del Pipeline

In [ ]:
X_train_transformado = preprocesador.transform(X_train)
X_test_transformado = preprocesador.transform(X_test)

# Algunos preprocesadores devuelven matrices sparse; SHAP trabaja mejor con arrays/DataFrames.
if hasattr(X_train_transformado, "toarray"):
    X_train_transformado = X_train_transformado.toarray()

if hasattr(X_test_transformado, "toarray"):
    X_test_transformado = X_test_transformado.toarray()

print("X_train transformado:", X_train_transformado.shape)
print("X_test transformado:", X_test_transformado.shape)

## 7. Obtencion de nombres de variables despues del OneHotEncoder

In [ ]:
nombres_variables = preprocesador.get_feature_names_out()

X_train_shap = pd.DataFrame(
    X_train_transformado,
    columns=nombres_variables,
    index=X_train.index,
)

X_test_shap = pd.DataFrame(
    X_test_transformado,
    columns=nombres_variables,
    index=X_test.index,
)

display(X_train_shap.head())
print("Cantidad de variables despues del preprocesamiento:", X_train_shap.shape[1])

## 8. Aplicacion de SHAP

SHAP es una tecnica de interpretabilidad que asigna a cada variable una contribucion a la prediccion del modelo. Se usa porque permite explicar tanto el comportamiento global del modelo como predicciones individuales. En Regresion Logistica, valores SHAP positivos aumentan el riesgo estimado de churn y valores negativos lo reducen.

In [ ]:
explainer = shap.LinearExplainer(modelo, X_train_shap)
shap_values = explainer(X_test_shap)

print("Valores SHAP calculados:", shap_values.values.shape)

## 9. Grafica de importancia global de variables

In [ ]:
plt.figure()
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("Importancia global de variables - SHAP")
plt.tight_layout()
ruta_importancia = RESULTADOS_DIR / "shap_importancia_global.png"
plt.savefig(ruta_importancia, dpi=300, bbox_inches="tight")
plt.close()

print("Grafica guardada en:", ruta_importancia)

## 10. Grafica beeswarm de SHAP

In [ ]:
plt.figure()
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("Distribucion del impacto de variables - SHAP")
plt.tight_layout()
ruta_beeswarm = RESULTADOS_DIR / "shap_beeswarm.png"
plt.savefig(ruta_beeswarm, dpi=300, bbox_inches="tight")
plt.close()

print("Grafica guardada en:", ruta_beeswarm)

## 11. Explicacion individual de un cliente con riesgo de churn

In [ ]:
probabilidades = pipeline.predict_proba(X_test)[:, 1]
posicion_cliente_alto_riesgo = int(np.argmax(probabilidades))
indice_cliente_alto_riesgo = X_test.index[posicion_cliente_alto_riesgo]

cliente_original = df.loc[[indice_cliente_alto_riesgo]].copy()
probabilidad_churn = probabilidades[posicion_cliente_alto_riesgo]
prediccion = pipeline.predict(X_test.iloc[[posicion_cliente_alto_riesgo]])[0]

print("Datos originales del cliente con mayor probabilidad de churn:")
display(cliente_original)
print("Probabilidad de churn:", probabilidad_churn)
print("Prediccion del modelo:", prediccion)

valores_shap_cliente = shap_values.values[posicion_cliente_alto_riesgo]
tabla_cliente = pd.DataFrame(
    {
        "Variable": X_test_shap.columns,
        "Valor_SHAP": valores_shap_cliente,
    }
)
tabla_cliente["Impacto"] = np.where(
    tabla_cliente["Valor_SHAP"] > 0,
    "Aumenta el riesgo de churn",
    "Reduce el riesgo de churn",
)
tabla_cliente = tabla_cliente.reindex(
    tabla_cliente["Valor_SHAP"].abs().sort_values(ascending=False).index
).head(10)

ruta_cliente = RESULTADOS_DIR / "shap_cliente_alto_riesgo.csv"
tabla_cliente.to_csv(ruta_cliente, index=False)

print("Variables que mas influyeron en la prediccion del cliente:")
display(tabla_cliente)
print("Tabla guardada en:", ruta_cliente)

## 12. Conclusiones de interpretabilidad

SHAP permitio interpretar el comportamiento del modelo de Regresion Logistica y entender que variables tienen mayor impacto en la prediccion de abandono. Las variables con valores SHAP positivos aumentan el riesgo de churn, mientras que las variables con valores SHAP negativos reducen ese riesgo.

En un contexto de churn, esta interpretacion es util porque ayuda a identificar factores accionables para retencion. Por ejemplo, tener contrato mensual puede aumentar el riesgo de churn, ser un cliente nuevo puede aumentar el riesgo de abandono y cargos altos tambien pueden elevar la probabilidad estimada de salida. En cambio, una mayor permanencia, contratos de mas largo plazo o mas servicios contratados pueden reducir el riesgo de churn.

Estos resultados pueden servir como base para disenar recomendaciones de retencion personalizadas, priorizando clientes con alta probabilidad de abandono y entendiendo las razones principales de esa prediccion.